# Notebook 99. Comparacion final
Settings: Accelerator **None**, Internet **Off**. Input: dataset privado **banana-resultados** con las 6 carpetas de `resultados/` del notebook 01 (kit v1.3, caché v3: test por días, n = 232).

In [ ]:
import os, glob, json, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.stats.contingency_tables import mcnemar

MODELS = ["resnet50", "efficientnet_b0", "mobilenetv2", "vit_b16", "swin_t", "coatnet_0"]
NAMES = {"resnet50": "ResNet50", "efficientnet_b0": "EfficientNetB0", "mobilenetv2": "MobileNetV2",
         "vit_b16": "ViT-B/16", "swin_t": "Swin-T", "coatnet_0": "CoAtNet-0"}
CLASES = ["Sigatoka", "Cordana", "Healthy"]
def f(pat):
    r = glob.glob(f"/kaggle/input/**/{pat}", recursive=True); assert len(r) == 1, (pat, r); return r[0]
P = {m: pd.read_csv(f(f"pred_test_{m}.csv")).set_index("sample_id").sort_index() for m in MODELS}
M = {m: json.load(open(f(f"metrics_{m}.json"))) for m in MODELS}
H = {m: pd.read_csv(f(f"history_{m}.csv")) for m in MODELS}
for m in MODELS:
    assert not M[m]["smoke_test"], f"{m} es la prueba de 1 epoca, no el entrenamiento completo"
    assert len(P[m]) == 232 and (P[m].index == P[MODELS[0]].index).all()
OUT = "/kaggle/working/consolidado"; os.makedirs(OUT, exist_ok=True)
print({m: (M[m]["env"]["gpu"], M[m]["env"]["torch"], M[m]["epochs_run"]) for m in MODELS})

In [ ]:
g = pd.DataFrame([{"Model": NAMES[m], "Accuracy (%)": 100 * M[m]["accuracy"],
                   "95% CI (%)": f"{100*M[m]['ci95_low']:.2f}--{100*M[m]['ci95_high']:.2f}",
                   "Precision (%)": 100 * M[m]["precision_macro"], "Recall (%)": 100 * M[m]["recall_macro"],
                   "F1-score (%)": 100 * M[m]["f1_macro"], "Parameters (M)": M[m]["params"] / 1e6,
                   "Epochs": M[m]["epochs_run"], "Training (min)": M[m]["train_minutes"],
                   "Inference (ms/image)": M[m]["inference_ms_per_image"]} for m in MODELS])
g.to_csv(f"{OUT}/tabla_global.csv", index=False)
pc = pd.DataFrame({NAMES[m]: dict(zip(CLASES, [100 * v for v in M[m]["f1"]])) for m in MODELS}).T
pc.to_csv(f"{OUT}/tabla_f1_por_clase.csv")
print(g.round(2).to_string(index=False)); print(pc.round(2))

def latex(df, cap, lab):
    head = " & ".join(r"\textbf{" + c.replace("%", r"\%") + "}" for c in df.columns) + r" \\"
    body = "\n".join(" & ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + r" \\" for r in df.values)
    return "\n".join([r"\begin{table}[H]", r"\caption{" + cap + r"\label{" + lab + "}}",
                      r"\begin{tabular}{l" + "r" * (df.shape[1] - 1) + "}", r"\toprule", head, r"\midrule",
                      body.replace("%", r"\%"), r"\bottomrule", r"\end{tabular}", r"\end{table}"])
open(f"{OUT}/tabla_rendimiento.tex", "w").write(latex(g[["Model", "Accuracy (%)", "95% CI (%)", "Precision (%)", "Recall (%)", "F1-score (%)"]],
     "Performance of the six models on the test set.", "tab:performance"))
open(f"{OUT}/tabla_costo.tex", "w").write(latex(g[["Model", "Parameters (M)", "Epochs", "Training (min)", "Inference (ms/image)"]],
     "Size and computational cost of the six models.", "tab:cost"))

In [ ]:
filas = []
for a, b in itertools.combinations(MODELS, 2):
    ca = P[a].y_pred == P[a].y_true; cb = P[b].y_pred == P[b].y_true
    t = [[int((ca & cb).sum()), int((ca & ~cb).sum())], [int((~ca & cb).sum()), int((~ca & ~cb).sum())]]
    rc = mcnemar(t, exact=False, correction=True)   # chi cuadrado con correccion, como en la 3.1
    rx = mcnemar(t, exact=True)                      # version exacta, solo como respaldo
    filas.append(dict(model_a=NAMES[a], model_b=NAMES[b], n01=t[0][1], n10=t[1][0], chi2=float(rc.statistic),
                      p_value=float(rc.pvalue), p_exact=float(rx.pvalue), significant=bool(rc.pvalue < .05)))
mc = pd.DataFrame(filas)
ALPHA, ALPHA_BONF = 0.05, 0.05 / len(filas)          # Bonferroni: 0.05 / 15
mc["significant_bonferroni"] = mc.p_value < ALPHA_BONF
mc.to_csv(f"{OUT}/tabla_mcnemar.csv", index=False)
pm = pd.DataFrame(np.nan, index=[NAMES[m] for m in MODELS], columns=[NAMES[m] for m in MODELS])
for r in filas:
    pm.loc[r["model_a"], r["model_b"]] = pm.loc[r["model_b"], r["model_a"]] = r["p_value"]
pm.to_csv(f"{OUT}/matriz_p_mcnemar.csv")
t = pd.DataFrame({"Model A": mc.model_a, "Model B": mc.model_b, "n01": mc.n01, "n10": mc.n10,
                  "Chi-square": mc.chi2.map(lambda v: f"{v:.2f}"), "p-value": mc.p_value.map(lambda v: f"{v:.4f}"),
                  "$p < 0.05$": mc.significant.map({True: "Yes", False: "No"}),
                  "$p < 0.0033$": mc.significant_bonferroni.map({True: "Yes", False: "No"})})
open(f"{OUT}/tabla_mcnemar.tex", "w").write(latex(t, "McNemar test between pairs of models on the test set (n01: images correct only for model A; n10: images correct only for model B; 0.0033 = Bonferroni threshold).", "tab:mcnemar"))
print(mc.round(4).to_string(index=False))

In [ ]:
plt.rcParams.update({"font.family": "serif", "font.size": 11})
vmax = max(np.array(M[m]["confusion"]).max() for m in MODELS)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, m in zip(axes.ravel(), MODELS):
    cm = np.array(M[m]["confusion"]); ax.imshow(cm, cmap="Greys", vmin=0, vmax=vmax)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > vmax / 2 else "black")
    ax.set_title(NAMES[m]); ax.set_xticks(range(3), CLASES); ax.set_yticks(range(3), CLASES)
    ax.set_xlabel("Predicted class"); ax.set_ylabel("Actual class")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_cm_6_modelos.pdf"); plt.close()

estilos = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1))]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for m, s in zip(MODELS, estilos):
    ax[0].plot(H[m].epoch, H[m].val_loss, color="black", linestyle=s, label=NAMES[m])
    ax[1].plot(H[m].epoch, H[m].val_acc, color="black", linestyle=s, label=NAMES[m])
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Validation loss")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Validation accuracy"); ax[1].legend(fontsize=9)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_curvas_validacion.pdf"); plt.close()

acc = np.array([100 * M[m]["accuracy"] for m in MODELS])
err = [acc - [100 * M[m]["ci95_low"] for m in MODELS], [100 * M[m]["ci95_high"] for m in MODELS] - acc]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([NAMES[m] for m in MODELS], acc, color="0.65", edgecolor="black", yerr=err, capsize=4)
ax.set_ylabel("Test accuracy (%)"); ax.set_ylim(max(0, acc.min() - 15), 100)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_accuracy_ic95.pdf"); plt.close()
print(sorted(os.listdir(OUT)))

In [ ]:
# Tablas por modelo (seccion 4.2) y resumen del entrenamiento
CLS = ["Sigatoka", "Cordana", "Healthy"]
LAB = {"resnet50": "resnet50", "efficientnet_b0": "efficientnetb0", "mobilenetv2": "mobilenetv2",
       "vit_b16": "vit", "swin_t": "swin", "coatnet_0": "coatnet"}
res = []
for m in MODELS:
    r = M[m]; cm = np.array(r["confusion"])
    t = pd.DataFrame({"Class": CLS, "Precision (%)": [f"{100*v:.2f}" for v in r["precision"]],
                      "Recall (%)": [f"{100*v:.2f}" for v in r["recall"]], "F1-score (%)": [f"{100*v:.2f}" for v in r["f1"]],
                      "Images": cm.sum(1).tolist()})
    t.loc[len(t)] = ["Macro average", f"{100*r['precision_macro']:.2f}", f"{100*r['recall_macro']:.2f}", f"{100*r['f1_macro']:.2f}", int(cm.sum())]
    cap = (f"Test results of {NAMES[m]} by class. Accuracy: {100*r['accuracy']:.2f}% "
           f"(95% Wilson CI: {100*r['ci95_low']:.2f}% to {100*r['ci95_high']:.2f}%).")
    open(f"{OUT}/tabla_clase_{m}.tex", "w").write(latex(t, cap.replace("%", "\\%"), f"tab:{LAB[m]}").replace("\\\\%", "\\%"))
    h = H[m]; b = int(h.val_loss.idxmin())
    res.append(dict(model=NAMES[m], epochs_run=len(h), best_epoch=int(h.epoch[b]), min_val_loss=round(float(h.val_loss[b]), 4),
                    val_acc_best=round(float(h.val_acc[b]), 4), train_acc_best=round(float(h.train_acc[b]), 4),
                    train_minutes=round(r["train_minutes"], 2), test_correct=int(np.trace(np.array(r["confusion"]))), test_n=int(np.array(r["confusion"]).sum())))
pd.DataFrame(res).to_csv(f"{OUT}/resumen_entrenamiento.csv", index=False)
print(pd.DataFrame(res).to_string(index=False))
print(sorted(os.listdir(OUT)))